In [10]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import kagglehub
import pandas as pd
import os
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModelForSeq2SeqLM, T5Tokenizer, T5ForConditionalGeneration

import OpenAttack as oa
import torch.nn.functional as F
from datasets import Dataset
import numpy as np
from tqdm import tqdm

In [2]:
path = kagglehub.dataset_download("babykrishnaml/unified-mental-health-nlp-dataset")

files = os.listdir(path)
csv_path = [f for f in files if f.endswith(".csv")][0]

df = pd.read_csv(os.path.join(path, csv_path))
df.head()

,text,label_emotion,label_topic,label_distortion,response,dataset_source
0,"Now if he does off himself, everyone will thin...",['neutral'],NaN,NaN,NaN,goemotions_split
1,WHY THE FUCK IS BAYLESS ISOING,['anger'],NaN,NaN,NaN,goemotions_split
2,To make her feel threatened,['fear'],NaN,NaN,NaN,goemotions_split
3,Dirty Southern Wankers,['annoyance'],NaN,NaN,NaN,goemotions_split
4,OmG pEyToN iSn'T gOoD eNoUgH tO hElP uS iN tHe...,['surprise'],NaN,NaN,NaN,goemotions_split


In [3]:
# Standardize
df.columns = [c.lower() for c in df.columns]

TEXT_COL = "text"
LABEL_COL = "label_emotion"

df = df.dropna(subset=[TEXT_COL, LABEL_COL])
# Binary mapping
def map_label(x):
    if x.lower() == "['neutral']":
        return 0
    else:
        return 1

df["label"] = df[LABEL_COL].apply(map_label)

df = df[[TEXT_COL, "label"]]
df.dropna(inplace=True)

In [4]:
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

train_ds = dataset["train"]
test_ds = dataset["test"]

In [5]:
def load_model(path):
    # detect model type from path
    if "Roberta" in path:
        tokenizer = AutoTokenizer.from_pretrained("roberta-base")
    else:
        tokenizer = AutoTokenizer.from_pretrained(
            path,
            local_files_only=True
        )

    model = AutoModelForSequenceClassification.from_pretrained(
        path,
        low_cpu_mem_usage=True,
        local_files_only=True
    )

    return model, tokenizer

In [35]:
class TransformerClassifier(oa.Classifier):
    def __init__(self, model, tokenizer, device="cpu"):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def get_prob(self, input_):
        inputs = self.tokenizer(
            input_,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)
        return outputs.logits.softmax(dim=1).cpu().numpy()

    def get_pred(self, input_):
        return self.get_prob(input_).argmax(axis=1)

In [36]:
attackers = {
    "textfooler": oa.attackers.TextFoolerAttacker(),
    "deepwordbug": oa.attackers.DeepWordBugAttacker(),
    "uat": oa.attackers.UATAttacker()
}

In [51]:
def run_attackeval(model, tokenizer, dataset, attacker, num_samples=50):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    clf = TransformerClassifier(model, tokenizer, device=device)
    attack_eval = oa.AttackEval(attacker,clf)
    results = attack_eval.eval(dataset, visualize=True)

In [6]:
trained_model = {}

trained_model["bert"] = (*load_model("./Bert"), test_ds)
trained_model["electra"] = (*load_model("./Electra"), test_ds)
trained_model["roberta"] = (*load_model("./Roberta"), test_ds)

In [8]:
formatted_data = [
    {"x": test_ds["text"][i], "y": test_ds["label"][i]}
    for i in range(min(50, len(test_ds["text"])))
]

In [ ]:
# without using any defense:

In [ ]:
all_results = {}

for name, (model, tokenizer, test_data) in trained_model.items():
    print(f"\n========== {name.upper()} ==========\n")

    for name, attacker in attackers.items():
        print(f"\n{name.upper()}")

        results = run_attackeval(
            model,
            tokenizer,
            formatted_data,
            attacker,
            num_samples=50 
        )


    all_results[name] = results


========== BERT ==========


TEXTFOOLER
Sample: 1 =====================================================================
Label: 1 (96.96%) --> 0 (52.27%)            |                                   
                                            |                                   
The bus    loop was never going to entice   |                                   
the jalopy loop was never going to entice   |                                   
                                            |                                   
families to transfer their kids to city     | Running Time:            0.37025  
families to transfer their kids to city     | Query Exceeded:          no       
                                            | Victim Model Queries:    82       
schools . Is it great for kids    already   | Succeed:                 yes      
schools . is it heavy for tiddler already   |                                   
                                            |                        

In [ ]:
# using sentence level defense - developed before midsem:

In [44]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
PARA_MODEL = "t5-small"

para_tokenizer = AutoTokenizer.from_pretrained(PARA_MODEL)
para_model = AutoModelForSeq2SeqLM.from_pretrained(PARA_MODEL).to(device)
para_model.eval()

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [46]:
def generate_paraphrases(text, num_return=3):
    input_text = "paraphrase: " + text + " </s>"

    encoding = para_tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    outputs = para_model.generate(
        **encoding,
        max_length=128,
        num_beams=5,
        num_return_sequences=num_return,
        temperature=1.5
    )

    paraphrases = [
        para_tokenizer.decode(out, skip_special_tokens=True)
        for out in outputs
    ]

    return paraphrases

In [62]:
class SmoothedClassifier(oa.Classifier):
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.cache = {}

    def get_prob(self, input_):
        all_probs = []

        for text in input_:
            if text in self.cache:
                paraphrases = self.cache[text]
            else:
                paraphrases = generate_paraphrases(text, num_return=5)
                self.cache[text] = paraphrases

            variants = [text] + paraphrases

            inputs = self.tokenizer(
                variants,
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).to(self.device)

            with torch.no_grad():
                outputs = self.model(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

            avg_prob = probs.mean(dim=0)
            all_probs.append(avg_prob.cpu().numpy())

        return np.array(all_probs)

    def get_pred(self, input_):
        return self.get_prob(input_).argmax(axis=1)
    

In [60]:
def run_attackeval_sentence_defense(model, tokenizer, dataset, attacker, num_samples=50):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    clf = SmoothedClassifier(model, tokenizer, device=device)
    attack_eval = oa.AttackEval(attacker,clf)
    results = attack_eval.eval(dataset, visualize=True)

In [63]:
for name, (model, tokenizer, test_data) in trained_model.items():
    print(f"\n========== {name.upper()} ==========\n")

    for name, attacker in attackers.items():
        print(f"\n{name.upper()}")

        results = run_attackeval_sentence_defense(
            model,
            tokenizer,
            formatted_data,
            attacker,
            num_samples=50 
        )


========== BERT ==========


TEXTFOOLER
Sample: 1 =====================================================================
Label: 0 (73.78%) --> Failed!               |                                   
                                            | Running Time:            0.82235  
The bus loop was never going to entice      | Query Exceeded:          no       
families to transfer their kids to city     | Victim Model Queries:    196      
schools . Is it great for kids already      | Succeed:                 no       
there ? Yes .                               |                                   
                                            |                                   
Sample: 2 =====================================================================
Label: 0 (66.17%) --> 1 (56.85%)            |                                   
                                            | Running Time:            0.12715  
in my self for failing so     hard          | Query Exceeded:         

In [ ]:
#new attack model

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
t5_model = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)
t5_model.eval()

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [13]:
def predict(model, tokenizer, texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
    
    return probs.cpu().numpy(), np.argmax(probs.cpu().numpy(), axis=1)

In [23]:
def generate_context(sentence, num_return=1):
    prompt = f"add a contrasting clause to this sentence without changing its main meaning: {sentence}"
    
    inputs = t5_tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    
    outputs = t5_model.generate(
        **inputs,
        max_length=128,
        num_return_sequences=num_return,
        do_sample=True,
        top_k=50,
        top_p=0.95
    )
    
    results = [
        t5_tokenizer.decode(o, skip_special_tokens=True)
        for o in outputs
    ]
    
    return results

In [24]:
def best_attack(sentence, label, model, tokenizer, tries=5):
    candidates = generate_context(sentence, num_return=tries)
    
    best_sentence = sentence
    best_score = 0
    
    for c in candidates:
        probs, _ = predict(model, tokenizer, [c])
        
        score = 1 - probs[0][label]  # reduce true class confidence
        
        if score > best_score:
            best_score = score
            best_sentence = c
    
    return best_sentence

In [25]:
def run_attack(model, tokenizer, samples, max_samples=50):
    success = 0
    total = 0
    
    for sample in tqdm(samples[:max_samples]):
        x = sample["x"]
        y = sample["y"]
        
        probs_orig, pred_orig = predict(model, tokenizer, [x])
        
        # skip incorrect originals
        if pred_orig[0] != y:
            continue
        
        # generate adversarial
        x_adv = best_attack(x, y, model, tokenizer)
        
        probs_adv, pred_adv = predict(model, tokenizer, [x_adv])
        
        if pred_adv[0] != y:
            success += 1
        
        total += 1
    
    return success / total if total > 0 else 0

In [26]:
models = {}
tokenizers = {}

for name, (model, tokenizer, test_data) in trained_model.items():    
    model.to(device)
    model.eval()
    
    models[name] = model
    tokenizers[name] = tokenizer


for name in models:
    print(f"\nAttacking {name.upper()}...")
    
    asr = run_attack(
        models[name],
        tokenizers[name],
        formatted_data,
        max_samples=50
    )
    
    print(f"Attack Success Rate (ASR): {asr:.4f}")


Attacking BERT...


  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [02:25<00:00,  2.91s/it]


Attack Success Rate (ASR): 0.5652

Attacking ELECTRA...


100%|██████████| 50/50 [02:21<00:00,  2.82s/it]


Attack Success Rate (ASR): 0.6889

Attacking ROBERTA...


100%|██████████| 50/50 [02:17<00:00,  2.76s/it]

Attack Success Rate (ASR): 0.7045
